# GDUT final test evaluation

Input: the saved preparation output, both Stage1 outputs, and A1 through A6 outputs. Run after validation-based model selection is complete. Output: test_evaluation/summary.csv, per-class AP, plots, and coverage metadata. The two Stage1 checkpoints represent the A0 controls.

Validation scores come from the saved best-checkpoint validation in each run_manifest.json. The A0 rows use the corresponding Stage1 run. Test scores come from evaluation on the GDUT test split. Both tables are displayed below.

Use a fresh Kaggle session. Enable Internet for package installation. The input image size is fixed at 640.


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "off",
        "--disable-pip-version-check",
        "-q",
        "ultralytics==8.4.115",
        "albumentations==2.0.8",
    ]
)


In [ ]:
import hashlib
import json
import os
from pathlib import Path

CODE_DIR = Path("/kaggle/working/research_code")
CODE_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE_DIR))
os.environ["PYTHONPATH"] = str(CODE_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["YOLO_CONFIG_DIR"] = "/kaggle/working/yolo_settings"


## Source modules

These cells write the importable modules used by the notebook and its distributed workers.


In [ ]:
%%writefile /kaggle/working/research_code/albumentations_hardcase_policy.py
"""Color-preserving hard-case augmentation for helmet detection."""

from __future__ import annotations

from pathlib import Path
from typing import Iterable

import cv2


def build_hardcase_transform(image_size: int = 640):
    """
    Build an Albumentations transform for helmet hard cases.

    Targeted effects:
    - RandomShadow: cloud/scaffolding shadows.
    - RandomBrightnessContrast: harsh sunlight and low light.
    - CoarseDropout: partial occlusion from scaffolding/grid structures.

    Bboxes use YOLO format: class_id x_center y_center width height.
    """
    import albumentations as A

    coarse_kwargs = {
        "num_holes_range": (1, 8),
        "hole_height_range": (0.03, 0.18),
        "hole_width_range": (0.03, 0.18),
        "fill": 0,
        "p": 0.35,
    }

    return A.Compose(
        [
            A.LongestMaxSize(max_size=image_size, p=1.0),
            A.PadIfNeeded(
                min_height=image_size,
                min_width=image_size,
                border_mode=cv2.BORDER_CONSTANT,
                fill=114,
                p=1.0,
            ),
            A.RandomShadow(
                shadow_roi=(0, 0, 1, 1),
                num_shadows_limit=(1, 3),
                shadow_dimension=5,
                p=0.35,
            ),
            A.RandomBrightnessContrast(
                brightness_limit=0.25, contrast_limit=0.25, p=0.45
            ),
            A.MotionBlur(blur_limit=5, p=0.12),
            A.GaussNoise(std_range=(0.02, 0.08), p=0.18),
            A.CoarseDropout(**coarse_kwargs),
        ],
        bbox_params=A.BboxParams(
            format="yolo", label_fields=["class_labels"], min_visibility=0.20
        ),
    )


def read_yolo_label(label_path: Path) -> tuple[list[list[float]], list[int]]:
    boxes: list[list[float]] = []
    labels: list[int] = []
    if not label_path.exists():
        return boxes, labels
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        labels.append(int(float(parts[0])))
        boxes.append([float(value) for value in parts[1:]])
    return boxes, labels


def write_yolo_label(
    label_path: Path, boxes: Iterable[Iterable[float]], labels: Iterable[int]
) -> None:
    lines = []
    for label, box in zip(labels, boxes):
        x, y, w, h = box
        x = min(1.0, max(0.0, float(x)))
        y = min(1.0, max(0.0, float(y)))
        w = min(1.0, max(0.0, float(w)))
        h = min(1.0, max(0.0, float(h)))
        if w <= 0 or h <= 0:
            continue
        lines.append(f"{int(label)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")


In [ ]:
%%writefile /kaggle/working/research_code/custom_ablation_modules.py
"""CoordConv, RepConv, routed attention, and Focal-EIoU modules."""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


def autopad(k: int, p: int | None = None, d: int = 1) -> int:
    """Return padding that preserves spatial shape for odd kernels."""
    if p is None:
        p = d * (k - 1) // 2
    return p


class ConvBNAct(nn.Module):
    """Convolution followed by batch normalization and SiLU."""

    def __init__(
        self,
        c1: int,
        c2: int,
        k: int = 1,
        s: int = 1,
        p: int | None = None,
        g: int = 1,
        d: int = 1,
        act: bool = True,
    ) -> None:
        super().__init__()
        self.conv = nn.Conv2d(
            c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False
        )
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.bn(self.conv(x)))


class CoordConv(nn.Module):
    """
    CoordConv layer: append normalized x/y coordinate channels before a conv.

    """

    def __init__(
        self, c1: int, c2: int, k: int = 3, s: int = 1, with_r: bool = False
    ) -> None:
        super().__init__()
        self.with_r = with_r
        extra = 3 if with_r else 2
        self.conv = ConvBNAct(c1 + extra, c2, k=k, s=s)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, _, h, w = x.shape
        yy = (
            torch.linspace(-1.0, 1.0, h, device=x.device, dtype=x.dtype)
            .view(1, 1, h, 1)
            .expand(b, 1, h, w)
        )
        xx = (
            torch.linspace(-1.0, 1.0, w, device=x.device, dtype=x.dtype)
            .view(1, 1, 1, w)
            .expand(b, 1, h, w)
        )
        coords = [xx, yy]
        if self.with_r:
            rr = torch.sqrt(torch.clamp(xx.square() + yy.square(), min=0.0))
            coords.append(rr)
        return self.conv(torch.cat([x, *coords], dim=1))


class RepConv(nn.Module):
    """
    RepConv block with 3x3, 1x1, and optional identity branches during training.

    Call `switch_to_deploy()` before ONNX/TensorRT export. The deploy path is a
    single 3x3 Conv2d plus activation.
    """

    def __init__(
        self,
        c1: int,
        c2: int,
        k: int = 3,
        s: int = 1,
        deploy: bool = False,
        act: bool = True,
    ) -> None:
        super().__init__()
        assert k == 3, "This RepConv implementation fuses to a 3x3 kernel."
        self.deploy = deploy
        self.in_channels = c1
        self.out_channels = c2
        self.stride = s
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

        if deploy:
            self.rbr_reparam = nn.Conv2d(c1, c2, 3, s, 1, bias=True)
        else:
            self.rbr_dense = nn.Sequential(
                nn.Conv2d(c1, c2, 3, s, 1, bias=False),
                nn.BatchNorm2d(c2),
            )
            self.rbr_1x1 = nn.Sequential(
                nn.Conv2d(c1, c2, 1, s, 0, bias=False),
                nn.BatchNorm2d(c2),
            )
            self.rbr_identity = nn.BatchNorm2d(c1) if c1 == c2 and s == 1 else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.deploy:
            return self.act(self.rbr_reparam(x))
        out = self.rbr_dense(x) + self.rbr_1x1(x)
        if self.rbr_identity is not None:
            out = out + self.rbr_identity(x)
        return self.act(out)

    @staticmethod
    def _fuse_conv_bn(
        branch: nn.Sequential | nn.BatchNorm2d, channels: int
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if isinstance(branch, nn.Sequential):
            conv = branch[0]
            bn = branch[1]
            kernel = conv.weight
        else:
            bn = branch
            input_dim = channels
            kernel = torch.zeros(
                (input_dim, input_dim, 3, 3),
                device=bn.weight.device,
                dtype=bn.weight.dtype,
            )
            for i in range(input_dim):
                kernel[i, i, 1, 1] = 1.0

        std = torch.sqrt(bn.running_var + bn.eps)
        t = (bn.weight / std).reshape(-1, 1, 1, 1)
        fused_kernel = kernel * t
        fused_bias = bn.bias - bn.running_mean * bn.weight / std
        return fused_kernel, fused_bias

    @staticmethod
    def _pad_1x1_to_3x3(kernel: torch.Tensor) -> torch.Tensor:
        if kernel.size(2) == 3:
            return kernel
        return F.pad(kernel, [1, 1, 1, 1])

    def get_equivalent_kernel_bias(self) -> tuple[torch.Tensor, torch.Tensor]:
        k3, b3 = self._fuse_conv_bn(self.rbr_dense, self.out_channels)
        k1, b1 = self._fuse_conv_bn(self.rbr_1x1, self.out_channels)
        if self.rbr_identity is None:
            kid = torch.zeros_like(k3)
            bid = torch.zeros_like(b3)
        else:
            kid, bid = self._fuse_conv_bn(self.rbr_identity, self.out_channels)
        return k3 + self._pad_1x1_to_3x3(k1) + kid.to(k3.device), b3 + b1 + bid.to(
            b3.device
        )

    def switch_to_deploy(self) -> None:
        if self.deploy:
            return
        kernel, bias = self.get_equivalent_kernel_bias()
        self.rbr_reparam = nn.Conv2d(
            self.in_channels,
            self.out_channels,
            3,
            self.stride,
            1,
            bias=True,
        )
        self.rbr_reparam.weight.data = kernel.detach().clone()
        self.rbr_reparam.bias.data = bias.detach().clone()
        del self.rbr_dense
        del self.rbr_1x1
        if hasattr(self, "rbr_identity"):
            del self.rbr_identity
        self.deploy = True


class BiFormerBlockLite(nn.Module):
    """
    Region routing and fine token attention with a residual connection.

    Region-level routing selects the key/value tokens used by fine attention.
    The residual block uses four heads and eight-pixel regions by default.
    """

    def __init__(
        self, channels: int, num_heads: int = 4, region_size: int = 8, topk: int = 4
    ) -> None:
        super().__init__()
        assert channels % num_heads == 0, "channels must be divisible by num_heads"
        self.channels = channels
        self.num_heads = num_heads
        self.region_size = region_size
        self.topk = topk
        self.qkv = nn.Conv2d(channels, channels * 3, 1, bias=False)
        self.proj = nn.Conv2d(channels, channels, 1, bias=False)
        self.norm = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        rs = self.region_size
        pad_h = (rs - h % rs) % rs
        pad_w = (rs - w % rs) % rs
        x_pad = F.pad(x, (0, pad_w, 0, pad_h))
        hp, wp = x_pad.shape[-2:]
        gh, gw = hp // rs, wp // rs

        q, k, v = self.qkv(x_pad).chunk(3, dim=1)
        q_regions = q.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()
        k_regions = k.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()
        v_regions = v.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()

        # [B, C, GH, GW, RS, RS] -> [B, R, T, C]
        q_tokens = q_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)
        k_tokens = k_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)
        v_tokens = v_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)

        q_region = q_tokens.mean(dim=2)
        k_region = k_tokens.mean(dim=2)
        route_logits = torch.matmul(q_region, k_region.transpose(-1, -2)) / (c**0.5)
        topk = min(self.topk, gh * gw)
        route_idx = route_logits.topk(topk, dim=-1).indices

        out_regions = []
        head_dim = c // self.num_heads
        for region_idx in range(gh * gw):
            selected = route_idx[:, region_idx]
            k_sel = torch.stack(
                [
                    k_tokens[bi, selected[bi]].reshape(topk * rs * rs, c)
                    for bi in range(b)
                ],
                dim=0,
            )
            v_sel = torch.stack(
                [
                    v_tokens[bi, selected[bi]].reshape(topk * rs * rs, c)
                    for bi in range(b)
                ],
                dim=0,
            )
            q_cur = q_tokens[:, region_idx]

            qh = q_cur.reshape(b, rs * rs, self.num_heads, head_dim).transpose(1, 2)
            kh = k_sel.reshape(b, topk * rs * rs, self.num_heads, head_dim).transpose(
                1, 2
            )
            vh = v_sel.reshape(b, topk * rs * rs, self.num_heads, head_dim).transpose(
                1, 2
            )
            attn = torch.softmax(
                torch.matmul(qh, kh.transpose(-1, -2)) / (head_dim**0.5), dim=-1
            )
            out = torch.matmul(attn, vh).transpose(1, 2).reshape(b, rs * rs, c)
            out_regions.append(out)

        y = torch.stack(out_regions, dim=1).reshape(b, gh, gw, rs, rs, c)
        y = y.permute(0, 5, 1, 3, 2, 4).reshape(b, c, hp, wp)
        y = y[:, :, :h, :w]
        return x + self.norm(self.proj(y))


def xywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    x, y, w, h = boxes.unbind(-1)
    half_w = w / 2
    half_h = h / 2
    return torch.stack((x - half_w, y - half_h, x + half_w, y + half_h), dim=-1)


def focal_eiou_loss(
    pred_boxes: torch.Tensor,
    target_boxes: torch.Tensor,
    xywh: bool = True,
    gamma: float = 0.5,
    eps: float = 1e-7,
    reduction: str = "mean",
) -> torch.Tensor:
    """
    Focal-EIoU box regression loss.

    Args:
        pred_boxes: [N, 4] predicted boxes.
        target_boxes: [N, 4] target boxes.
        xywh: True when boxes are center-x, center-y, width, height.
        gamma: focal weighting exponent.
    """
    if xywh:
        pred = xywh_to_xyxy(pred_boxes)
        target = xywh_to_xyxy(target_boxes)
    else:
        pred = pred_boxes
        target = target_boxes

    px1, py1, px2, py2 = pred.unbind(-1)
    tx1, ty1, tx2, ty2 = target.unbind(-1)
    pw = (px2 - px1).clamp(min=eps)
    ph = (py2 - py1).clamp(min=eps)
    tw = (tx2 - tx1).clamp(min=eps)
    th = (ty2 - ty1).clamp(min=eps)

    inter_x1 = torch.maximum(px1, tx1)
    inter_y1 = torch.maximum(py1, ty1)
    inter_x2 = torch.minimum(px2, tx2)
    inter_y2 = torch.minimum(py2, ty2)
    inter = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
    union = pw * ph + tw * th - inter + eps
    iou = (inter / union).clamp(min=eps, max=1.0)

    pcx = (px1 + px2) / 2
    pcy = (py1 + py2) / 2
    tcx = (tx1 + tx2) / 2
    tcy = (ty1 + ty2) / 2
    center_dist = (pcx - tcx).square() + (pcy - tcy).square()

    cw = (torch.maximum(px2, tx2) - torch.minimum(px1, tx1)).clamp(min=eps)
    ch = (torch.maximum(py2, ty2) - torch.minimum(py1, ty1)).clamp(min=eps)
    c2 = cw.square() + ch.square() + eps

    eiou = (
        1.0
        - iou
        + center_dist / c2
        + (pw - tw).square() / (cw.square() + eps)
        + (ph - th).square() / (ch.square() + eps)
    )
    loss = iou.pow(gamma) * eiou

    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    if reduction == "none":
        return loss
    raise ValueError(f"Unsupported reduction: {reduction}")


In [ ]:
%%writefile /kaggle/working/research_code/dataset_pipeline.py
"""Dataset conversion, split validation, and offline augmentation."""

from pathlib import Path
import collections
import hashlib
import json
import random
import shutil
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image

NAMES = {"gdut": ["blue", "white", "yellow", "red", "none"]}
SEED = 3407


def dump(path, obj):
    Path(path).write_text(
        json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8"
    )


def image_info(path):
    # Decode pixels before hashing to detect unreadable images.
    with Image.open(path) as im:
        im = im.convert("RGB")
        im.load()
        return im.size, hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest()


def valid_boxes(boxes, path):
    for cls, x, y, w, h in boxes:
        if cls != int(cls) or not 0 <= cls < 5 or not np.isfinite([x, y, w, h]).all():
            raise ValueError(f"Invalid class/nonfinite label: {path}: {cls,x,y,w,h}")
        if (
            w <= 0
            or h <= 0
            or min(x - w / 2, y - h / 2) < -1e-6
            or max(x + w / 2, y + h / 2) > 1 + 1e-6
        ):
            raise ValueError(f"Invalid normalized box: {path}: {cls,x,y,w,h}")
    return boxes


def read_gdut(root):
    ann = root / "Annotations"
    imgs = root / "JPEGImages"
    if not imgs.is_dir():
        raise FileNotFoundError(f"Expected VOC JPEGImages: {imgs}")
    trainval = (root / "ImageSets/Main/trainval.txt").read_text().split()
    test = (root / "ImageSets/Main/test.txt").read_text().split()
    assert len(set(trainval)) == len(trainval) and len(set(test)) == len(test)
    assert not set(trainval) & set(test), "Official split overlap"
    files = sorted(ann.glob("*.xml"))
    assert {p.stem for p in files} == set(trainval + test), "Unlisted/missing XML"
    records = []
    for index, p in enumerate(files):
        if index % 100 == 0:
            print(f"[SCAN GDUT] {index}/{len(files)} images decoded/hashed", flush=True)
        t = ET.parse(p).getroot()
        image = imgs / (p.stem + ".jpg")
        (w, h), digest = image_info(image)
        assert (w, h) == (
            int(t.findtext("size/width")),
            int(t.findtext("size/height")),
        ), p
        boxes = []
        for obj in t.findall("object"):
            name = obj.findtext("name").strip()
            if name not in NAMES["gdut"]:
                raise ValueError(f"Unknown GDUT class {name}: {p}")
            b = obj.find("bndbox")
            x1, y1, x2, y2 = [
                float(b.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax")
            ]
            # VOC is 1-based inclusive. Permit zero-valued lower edge found in some exports.
            x1 = max(0, x1 - 1)
            y1 = max(0, y1 - 1)
            x2 = min(w, x2)
            y2 = min(h, y2)
            boxes.append(
                [
                    NAMES["gdut"].index(name),
                    (x1 + x2) / (2 * w),
                    (y1 + y2) / (2 * h),
                    (x2 - x1) / w,
                    (y2 - y1) / h,
                ]
            )
        records.append(
            dict(
                source=str(image),
                relative=image.relative_to(root).as_posix(),
                boxes=valid_boxes(boxes, p),
                pixel_hash=digest,
                width=w,
                height=h,
                group=p.stem,
                official="test" if p.stem in set(test) else "trainval",
            )
        )
    assert len(records) == 3174, "Dataset version changed; review before training"
    return records


def split_records(records, key, audit=None):
    if key != "gdut":
        raise ValueError("Only GDUT is supported.")
    rng = random.Random(SEED)
    records = [dict(r) for r in records]
    test_by_hash = collections.defaultdict(list)
    for r in records:
        if r["official"] == "test":
            test_by_hash[r["pixel_hash"]].append(r.get("relative", r["group"]))
    excluded = [
        r
        for r in records
        if r["official"] == "trainval" and r["pixel_hash"] in test_by_hash
    ]
    if audit is not None:
        audit.update(
            policy="preserve_all_official_test_remove_overlapping_trainval",
            excluded_trainval=[
                dict(
                    relative=r.get("relative", r["group"]),
                    pixel_hash=r["pixel_hash"],
                    matching_test=test_by_hash[r["pixel_hash"]],
                )
                for r in excluded
            ],
            official_test_count=sum(r["official"] == "test" for r in records),
        )
    records = [
        r
        for r in records
        if not (r["official"] == "trainval" and r["pixel_hash"] in test_by_hash)
    ]
    print(
        f"[DEDUP GDUT] Excluded {len(excluded)} trainval images matching official test; test unchanged.",
        flush=True,
    )
    # Identical decoded images are one indivisible group, even if filenames differ.
    parent = {r["group"]: r["group"] for r in records}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    seen = {}
    for r in records:
        g = r["group"]
        old = seen.setdefault(r["pixel_hash"], g)
        parent[find(g)] = find(old)
    for r in records:
        r["group"] = find(r["group"])
    groups = collections.defaultdict(list)
    for i, r in enumerate(records):
        groups[r["group"]].append(i)
    for ids in groups.values():
        if len({records[i]["official"] for i in ids}) > 1:
            raise ValueError(
                "Exact image duplicate crosses official GDUT test/trainval; inspect before training."
            )
    candidates = [
        g for g, ids in groups.items() if records[ids[0]]["official"] == "trainval"
    ]
    rng.shuffle(candidates)
    target = round(sum(len(groups[g]) for g in candidates) * 0.2)
    val = set()
    n = 0
    for g in candidates:
        if n >= target:
            break
        val.add(g)
        n += len(groups[g])
    assignment = {
        g: (
            "test"
            if records[ids[0]]["official"] == "test"
            else "val" if g in val else "train"
        )
        for g, ids in groups.items()
    }
    for r in records:
        r["split"] = assignment[r["group"]]
    for s in ["train", "val", "test"]:
        classes = {int(b[0]) for r in records if r["split"] == s for b in r["boxes"]}
        assert classes == set(range(5)), f"{s} missing classes: {classes}"
    return records


def prepare(key, root, out, aug_image_size=640):
    if key != "gdut":
        raise ValueError("Only GDUT is supported.")
    root, out = Path(root), Path(out)
    if aug_image_size != 640:
        raise ValueError("The augmentation image size must be 640.")
    if out.exists():
        raise FileExistsError(
            f"Refusing to mix/rewrite an existing prepared dataset: {out}"
        )
    duplicate_audit = {}
    records = split_records(read_gdut(root), key, duplicate_audit)
    out.mkdir(parents=True)
    dump(out / "duplicate_audit.json", duplicate_audit)
    for r in records:
        uid = hashlib.sha256(r["relative"].encode()).hexdigest()[:20]
        r["id"] = uid
        dst = out / "base/images" / r["split"] / (uid + ".jpg")
        dst.parent.mkdir(parents=True, exist_ok=True)
        # Copy to writable Kaggle output; leave original image pixels unchanged.
        shutil.copy2(r["source"], dst)
        lab = out / "base/labels" / r["split"] / (uid + ".txt")
        lab.parent.mkdir(parents=True, exist_ok=True)
        lab.write_text(
            "".join(
                f"{int(b[0])} " + " ".join(f"{x:.10f}" for x in b[1:]) + "\n"
                for b in r["boxes"]
            )
        )
    manifest = {
        "dataset": key,
        "names": NAMES[key],
        "seed": SEED,
        "records": records,
        "preparation_version": 2,
        "augmentation_imgsz": aug_image_size,
        "duplicate_audit": duplicate_audit,
        "split_counts": dict(collections.Counter(r["split"] for r in records)),
        "box_counts": {
            s: dict(
                collections.Counter(
                    NAMES[key][int(b[0])]
                    for r in records
                    if r["split"] == s
                    for b in r["boxes"]
                )
            )
            for s in ["train", "val", "test"]
        },
        "group_assignment": {r["group"]: r["split"] for r in records},
        "notes": "GDUT test unchanged; overlapping trainval copies excluded and audited BEFORE remaining trainval split 80/20. All retained objects, including VOC difficult, retained.",
    }
    dump(out / "dataset_manifest.json", manifest)
    build_aug(out, aug_image_size)
    dump(
        out / "preparation_complete.json",
        {
            "complete": True,
            "records": len(records),
            "augmentation_imgsz": aug_image_size,
        },
    )
    return manifest


def build_aug(out, image_size=640):
    import cv2
    from albumentations_hardcase_policy import (
        build_hardcase_transform,
        read_yolo_label,
        write_yolo_label,
    )

    out = Path(out)
    transform = build_hardcase_transform(image_size)
    for i, image in enumerate(sorted((out / "base/images/train").glob("*.jpg"))):
        if i % 100 == 0:
            print(f"[HARD AUG] {i} images completed", flush=True)
        transform.set_random_seed(SEED + i)
        lab = out / "base/labels/train" / (image.stem + ".txt")
        boxes, classes = read_yolo_label(lab)
        rgb = cv2.cvtColor(cv2.imread(str(image)), cv2.COLOR_BGR2RGB)
        result = transform(image=rgb, bboxes=boxes, class_labels=classes)
        dst = out / "hardaug/images/train" / (image.stem + "_hardaug.jpg")
        dst.parent.mkdir(parents=True, exist_ok=True)
        assert cv2.imwrite(str(dst), cv2.cvtColor(result["image"], cv2.COLOR_RGB2BGR))
        write_yolo_label(
            out / "hardaug/labels/train" / (image.stem + "_hardaug.txt"),
            result["bboxes"],
            result["class_labels"],
        )
    dump(
        out / "augmentation_manifest.json",
        {
            "seed": SEED,
            "image_size": image_size,
            "per_image_seed": "3407 + index in sorted hashed image names",
            "copies_per_train_image": 1,
            "hue_saturation_transform_removed": True,
            "validation_test_augmented": False,
        },
    )


def mounted_yaml(prepared, augmented=False):
    import yaml

    prepared = Path(prepared)
    meta = json.loads((prepared / "dataset_manifest.json").read_text())
    work = Path("/kaggle/working/config")
    work.mkdir(parents=True, exist_ok=True)
    data = {
        "path": str(prepared),
        "names": dict(enumerate(meta["names"])),
        "train": [str(prepared / "base/images/train")]
        + ([str(prepared / "hardaug/images/train")] if augmented else []),
        "val": str(prepared / "base/images/val"),
        "test": str(prepared / "base/images/test"),
    }
    path = work / (meta["dataset"] + ("_aug" if augmented else "_base") + ".yaml")
    path.write_text(yaml.safe_dump(data, allow_unicode=True))
    return str(path)


def find_prepared(key, explicit=""):
    paths = (
        [Path(explicit)]
        if explicit
        else [p.parent for p in Path("/kaggle/input").rglob("dataset_manifest.json")]
    )
    paths = [
        p
        for p in paths
        if (p / "dataset_manifest.json").exists()
        and json.loads((p / "dataset_manifest.json").read_text())["dataset"] == key
    ]
    if len(paths) != 1:
        raise RuntimeError(f"Set PREPARED_DIR explicitly: found {paths}")
    meta = json.loads((paths[0] / "dataset_manifest.json").read_text())
    if (
        meta.get("preparation_version", 1) >= 2
        and not (paths[0] / "preparation_complete.json").exists()
    ):
        raise RuntimeError(
            "Prepared output is incomplete: hard augmentation did not finish. Rerun preparation in a fresh output."
        )
    return paths[0]


In [ ]:
%%writefile /kaggle/working/research_code/plain_logging.py
"""Plain-text formatting for Ultralytics console messages."""

import logging
import re
import sys
from functools import wraps


ANSI_ESCAPE = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
DECORATIONS = re.compile(r"[\U0001f000-\U0001faff\u2600-\u27bf\ufe0f\u200d]")
REPLACEMENTS = str.maketrans(
    {
        "\u2192": "->",
        "\u2190": "<-",
        "\u2194": "<->",
        "\u2265": ">=",
        "\u2264": "<=",
        "\u00d7": "x",
        "\u2705": "[OK]",
        "\u2713": "[OK]",
        "\u274c": "[FAIL]",
        "\u2717": "[FAIL]",
    }
)


def plain_text(message):
    """Remove console decoration while preserving labels and numeric values."""
    message = ANSI_ESCAPE.sub("", message).translate(REPLACEMENTS)
    return DECORATIONS.sub("", message)


class PlainFormatter(logging.Formatter):
    def format(self, record):
        message = plain_text(super().format(record))
        if record.levelno >= logging.WARNING and not message.startswith(
            record.levelname + ":"
        ):
            message = re.sub(r"^" + record.levelname + r"\s+", "", message)
            message = f"{record.levelname}: {message}"
        return message


class ProgressStream:
    """Write progress updates as complete, undecorated notebook lines."""

    def __init__(self, stream):
        self.stream = stream

    def write(self, message):
        cleaned = plain_text(message)
        cleaned = re.sub(r"[\u2500-\u259f]", "-", cleaned)
        if "\r" in cleaned:
            cleaned = cleaned.replace("\r", "").rstrip()
            if cleaned:
                cleaned += "\n"
        self.stream.write(cleaned)
        return len(message)

    def flush(self):
        self.stream.flush()

    def __getattr__(self, name):
        return getattr(self.stream, name)


def configure_progress_output():
    """Format the pinned Ultralytics progress writer without changing iteration."""
    from ultralytics.utils import TQDM

    if getattr(TQDM.__init__, "_plain_progress", False):
        return
    original = TQDM.__init__

    @wraps(original)
    def initialize(self, *args, **kwargs):
        if len(args) > 4:
            args = list(args)
            args[4] = ProgressStream(args[4] or sys.stdout)
        else:
            kwargs["file"] = ProgressStream(kwargs.get("file") or sys.stdout)
        original(self, *args, **kwargs)
        self.mininterval = max(self.mininterval, 10.0)

    initialize._plain_progress = True
    TQDM.__init__ = initialize


def configure_plain_logging():
    """Keep the library's log levels and handlers, using plain-text output."""
    from ultralytics.utils import LOGGER

    for handler in LOGGER.handlers:
        handler.setFormatter(PlainFormatter("%(message)s"))
    configure_progress_output()


In [ ]:
%%writefile /kaggle/working/research_code/preflight.py
"""Synthetic checks for shapes, losses, gradients, checkpoints, and fusion."""

from copy import deepcopy
from pathlib import Path
from types import SimpleNamespace
import tempfile
import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.cfg import DEFAULT_CFG_DICT
from research_trainer import (
    ResearchDetectionModel,
    patch,
    inventory,
    FocalEIoUBboxLoss,
    SPPFWithBiFormer,
)
from custom_ablation_modules import RepConv


def main():
    assert ultralytics.__version__ == "8.4.115", ultralytics.__version__
    torch.set_num_threads(2)
    torch.manual_seed(3407)
    reports = []
    for family in ["yolo11s", "yolov8s"]:
        for variant in ["A1", "A2", "A3", "A4", "A5", "A6"]:
            model = ResearchDetectionModel(family + ".yaml", nc=5, verbose=False)
            patch(model, variant)
            model.args = SimpleNamespace(**DEFAULT_CFG_DICT)
            if variant in ("A5", "A6"):
                expected_index = 10 if family == "yolo11s" else 9
                assert inventory(model)["BiFormerBlockLite"] == [expected_index]
                if family == "yolov8s":
                    assert isinstance(model.model[9], SPPFWithBiFormer)
            model.train()
            batch = {
                "img": torch.rand(2, 3, 64, 64),
                "batch_idx": torch.tensor([0.0, 1.0]),
                "cls": torch.tensor([[0.0], [4.0]]),
                "bboxes": torch.tensor([[0.5, 0.5, 0.3, 0.3], [0.5, 0.5, 0.4, 0.4]]),
            }
            loss, items = model(batch)
            assert torch.isfinite(loss).all(), (family, variant, loss)
            loss.sum().backward()
            assert isinstance(model.criterion.bbox_loss, FocalEIoUBboxLoss) == (
                variant in ["A4", "A6"]
            )
            for kind, ids in inventory(model).items():
                for i in ids:
                    grads = [
                        p.grad for p in model.model[i].parameters() if p.requires_grad
                    ]
                    assert grads and all(
                        g is not None and torch.isfinite(g).all() for g in grads
                    ), (family, variant, kind, i)
                    assert sum(g.abs().sum().item() for g in grads) > 0, (
                        family,
                        variant,
                        kind,
                        i,
                        "zero gradients",
                    )
            model.eval()
            model.criterion = None
            with tempfile.TemporaryDirectory(prefix="preflight_", dir=Path.cwd()) as td:
                path = Path(td) / "model.pt"
                torch.save(
                    {"model": deepcopy(model), "train_args": dict(DEFAULT_CFG_DICT)},
                    path,
                )
                restored = YOLO(str(path)).model
                assert inventory(restored) == inventory(model)
                with torch.no_grad():
                    before = model(batch["img"])
                    after = restored(batch["img"])
                    before = before[0] if isinstance(before, tuple) else before
                    after = after[0] if isinstance(after, tuple) else after
                torch.testing.assert_close(before, after, rtol=1e-4, atol=1e-4)
                assert before.shape == (2, 9, 84), before.shape
            reports.append(
                {
                    "model": family,
                    "variant": variant,
                    "modules": inventory(model),
                    "loss": float(loss.sum()),
                    "status": model.research_status,
                }
            )
            print("Validated configuration:", reports[-1], flush=True)
            del model, restored
    rep = RepConv(8, 8).eval()
    x = torch.randn(2, 8, 16, 16)
    with torch.no_grad():
        a = rep(x)
        rep.switch_to_deploy()
        b = rep(x)
    torch.testing.assert_close(a, b, rtol=1e-4, atol=1e-5)
    print("Architecture, gradient, checkpoint, and fusion checks completed.")
    return reports


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/research_code/pretrained_conversion.py
"""Function-preserving Conv conversions; module topology is unchanged."""

import torch
from custom_ablation_modules import CoordConv, RepConv, BiFormerBlockLite


def copy_bn(source, target):
    target.load_state_dict(source.state_dict())
    # BatchNorm state_dict excludes eps and momentum.
    target.eps = source.eps
    target.momentum = source.momentum


@torch.no_grad()
def coord_from_conv(source):
    c = source.conv
    if c.groups != 1 or c.bias is not None:
        raise ValueError("Expected unfused standard Conv + BN")
    new = CoordConv(
        c.in_channels, c.out_channels, k=c.kernel_size[0], s=c.stride[0], with_r=False
    )
    new.to(device=c.weight.device, dtype=c.weight.dtype)
    new.conv.conv.weight.zero_()
    new.conv.conv.weight[:, : c.in_channels].copy_(c.weight)
    copy_bn(source.bn, new.conv.bn)
    return new


@torch.no_grad()
def rep_from_conv(source):
    c = source.conv
    if c.groups != 1 or c.bias is not None or c.kernel_size != (3, 3):
        raise ValueError("Expected unfused standard 3x3 Conv + BN")
    new = RepConv(c.in_channels, c.out_channels, k=3, s=c.stride[0], deploy=False)
    new.to(device=c.weight.device, dtype=c.weight.dtype)
    new.rbr_dense[0].weight.copy_(c.weight)
    copy_bn(source.bn, new.rbr_dense[1])
    # Keep new branches trainable; zero BN scale makes their initial contribution zero.
    new.rbr_1x1[1].weight.zero_()
    new.rbr_1x1[1].bias.zero_()
    if new.rbr_identity is not None:
        new.rbr_identity.weight.zero_()
        new.rbr_identity.bias.zero_()
    return new


@torch.no_grad()
def identity_start_attention(channels, source):
    new = BiFormerBlockLite(channels=channels, num_heads=4)
    p = next(source.parameters())
    new.to(device=p.device, dtype=p.dtype)
    new.norm.weight.zero_()
    new.norm.bias.zero_()
    # Zero residual scale starts the attention block as an identity map.
    return new


In [ ]:
%%writefile /kaggle/working/research_code/research_trainer.py
"""Model construction and distributed trainers for the ablation experiments."""

import json
import os

from plain_logging import configure_plain_logging

configure_plain_logging()

from torch import nn
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.loss import BboxLoss, v8DetectionLoss

from custom_ablation_modules import (
    BiFormerBlockLite,
    CoordConv,
    RepConv,
    focal_eiou_loss,
)
from pretrained_conversion import (
    coord_from_conv,
    identity_start_attention,
    rep_from_conv,
)

ARCHITECTURE_REVISION = "biformer_both_backbones_v2"
INITIALIZATION_POLICY = "preserve_pretrained_v1"


class SPPFWithBiFormer(nn.Module):
    """Retain the YOLOv8 SPPF and apply attention to its P5 output."""

    def __init__(self, sppf, attention):
        super().__init__()
        self.sppf = sppf
        self.attention = attention

    def forward(self, x):
        return self.attention(self.sppf(x))


class FocalEIoUBboxLoss(BboxLoss):
    """Use Focal-EIoU for box regression and retain the native DFL term."""

    def forward(
        self,
        pred_dist,
        pred_bboxes,
        anchor_points,
        target_bboxes,
        target_scores,
        target_scores_sum,
        fg_mask,
        imgsz,
        stride,
    ):
        _, loss_dfl = super().forward(
            pred_dist,
            pred_bboxes,
            anchor_points,
            target_bboxes,
            target_scores,
            target_scores_sum,
            fg_mask,
            imgsz,
            stride,
        )
        weight = target_scores[fg_mask].sum(-1)
        loss_box = focal_eiou_loss(
            pred_bboxes[fg_mask],
            target_bboxes[fg_mask],
            xywh=False,
            gamma=0.5,
            reduction="none",
        )
        return (loss_box * weight).sum() / target_scores_sum, loss_dfl


class ResearchDetectionModel(DetectionModel):
    def init_criterion(self):
        criterion = v8DetectionLoss(self)
        if self.research_variant in ("A4", "A6"):
            criterion.bbox_loss = FocalEIoUBboxLoss(self.model[-1].reg_max).to(
                next(self.parameters()).device
            )
        return criterion


def inventory(model):
    """Return top-level graph indices containing each custom module."""
    module_types = {
        "CoordConv": CoordConv,
        "RepConv": RepConv,
        "BiFormerBlockLite": BiFormerBlockLite,
    }
    return {
        name: [
            index
            for index, layer in enumerate(model.model)
            if any(isinstance(child, module_type) for child in layer.modules())
        ]
        for name, module_type in module_types.items()
    }


def replace(model, index, replacement):
    previous = model.model[index]
    replacement.i = getattr(previous, "i", index)
    replacement.f = getattr(previous, "f", -1)
    replacement.type = replacement.__class__.__name__
    replacement.np = sum(parameter.numel() for parameter in replacement.parameters())
    model.model[index] = replacement


def patch(model, variant, initialization=INITIALIZATION_POLICY):
    """Construct one ablation from a stock Stage1 model."""
    if variant not in ("A1", "A2", "A3", "A4", "A5", "A6"):
        raise ValueError(f"Unknown variant: {variant}")
    if initialization not in (INITIALIZATION_POLICY, "legacy_random"):
        raise ValueError(f"Unknown initialization policy: {initialization}")
    preserve = initialization == INITIALIZATION_POLICY
    classes = {layer.__class__.__name__ for layer in model.model}
    if "C2PSA" in classes:
        family = "yolo11"
    elif "C2f" in classes:
        family = "yolov8"
    else:
        raise ValueError("The ablations support YOLO11s and YOLOv8s only.")

    conversions = []
    if variant in ("A2", "A6"):
        previous = model.model[0]
        conv = previous.conv
        replacement = (
            coord_from_conv(previous)
            if preserve
            else CoordConv(conv.in_channels, conv.out_channels, k=3, s=2)
        )
        replace(model, 0, replacement)
        conversions.append(
            {"index": 0, "module": "CoordConv", "copied_conv_bn": preserve}
        )

    if variant in ("A3", "A6"):
        for index, layer in enumerate(model.model):
            if variant == "A6" and index == 0:
                continue
            conv = getattr(layer, "conv", None)
            if getattr(conv, "kernel_size", None) == (3, 3):
                replacement = (
                    rep_from_conv(layer)
                    if preserve
                    else RepConv(
                        conv.in_channels,
                        conv.out_channels,
                        k=3,
                        s=conv.stride[0],
                    )
                )
                replace(model, index, replacement)
                conversions.append(
                    {"index": index, "module": "RepConv", "copied_conv_bn": preserve}
                )

    if variant in ("A5", "A6"):
        index = 10 if family == "yolo11" else 9
        previous = model.model[index]
        expected_type = "C2PSA" if family == "yolo11" else "SPPF"
        if previous.__class__.__name__ != expected_type:
            raise ValueError(f"Expected {expected_type} at layer {index}.")
        channels = (
            previous.cv1.conv.in_channels
            if family == "yolo11"
            else previous.cv2.conv.out_channels
        )
        attention = (
            identity_start_attention(channels, previous)
            if preserve
            else BiFormerBlockLite(channels, num_heads=4)
        )
        replacement = (
            attention if family == "yolo11" else SPPFWithBiFormer(previous, attention)
        )
        replace(model, index, replacement)
        conversions.append(
            {
                "index": index,
                "module": "BiFormerBlockLite",
                "placement": "replace_C2PSA" if family == "yolo11" else "after_SPPF",
                "identity_start": preserve,
            }
        )

    rep_indices = (
        [0, 1, 3, 5, 7, 17, 20] if family == "yolo11" else [0, 1, 3, 5, 7, 16, 19]
    )
    expected = {
        "CoordConv": [0] if variant in ("A2", "A6") else [],
        "RepConv": rep_indices if variant in ("A3", "A6") else [],
        "BiFormerBlockLite": (
            [10 if family == "yolo11" else 9] if variant in ("A5", "A6") else []
        ),
    }
    if variant == "A6":
        expected["RepConv"] = expected["RepConv"][1:]
    if inventory(model) != expected:
        raise RuntimeError(f"Unexpected module inventory: {inventory(model)}")

    model.research_variant = variant
    model.research_family = family
    model.research_status = "as_defined"
    model.architecture_revision = ARCHITECTURE_REVISION
    model.initialization_policy = initialization
    model.initialization_audit = conversions
    return model


class ResearchTrainer(DetectionTrainer):
    variant = "A1"

    def get_model(self, cfg=None, weights=None, verbose=True):
        # Each DDP worker constructs the graph before loading custom weights.
        model = self.set_model_names_for_load(
            ResearchDetectionModel(
                cfg,
                nc=self.data["nc"],
                ch=self.data["channels"],
                verbose=verbose,
            )
        )
        if weights is not None and hasattr(weights, "research_variant"):
            if weights.research_variant != self.variant:
                raise ValueError("Each ablation must start from its Stage1 checkpoint.")
            if (
                weights.research_family == "yolov8"
                and self.variant in ("A5", "A6")
                and getattr(weights, "architecture_revision", None)
                != ARCHITECTURE_REVISION
            ):
                raise ValueError(
                    "This YOLOv8 checkpoint predates the BiFormer extension."
                )
            patch(
                model,
                self.variant,
                initialization=getattr(
                    weights, "initialization_policy", "legacy_random"
                ),
            )
            model.load(weights)
        else:
            if weights is not None:
                model.load(weights)
            patch(model, self.variant)

        report = {
            "variant": self.variant,
            "family": model.research_family,
            "status": model.research_status,
            "rank": os.environ.get("RANK", "-1"),
            "modules": inventory(model),
            "nc": self.data["nc"],
            "box_loss": "Focal-EIoU" if self.variant in ("A4", "A6") else "CIoU",
            "cls_loss": "BCE",
            "dfl": "native Ultralytics DFL",
            "parameters": sum(parameter.numel() for parameter in model.parameters()),
            "architecture_revision": ARCHITECTURE_REVISION,
            "initialization_policy": model.initialization_policy,
            "initialization_audit": model.initialization_audit,
        }
        self.save_dir.mkdir(parents=True, exist_ok=True)
        path = self.save_dir / f"architecture_rank_{report['rank']}.json"
        path.write_text(json.dumps(report, indent=2), encoding="utf-8")
        print(json.dumps(report, indent=2), flush=True)
        return model


class A1Trainer(ResearchTrainer):
    variant = "A1"


class A2Trainer(ResearchTrainer):
    variant = "A2"


class A3Trainer(ResearchTrainer):
    variant = "A3"


class A4Trainer(ResearchTrainer):
    variant = "A4"


class A5Trainer(ResearchTrainer):
    variant = "A5"


class A6Trainer(ResearchTrainer):
    variant = "A6"


TRAINERS = {
    trainer.variant: trainer
    for trainer in (A1Trainer, A2Trainer, A3Trainer, A4Trainer, A5Trainer, A6Trainer)
}


In [ ]:
%%writefile /kaggle/working/research_code/run_experiment.py
"""Training and evaluation entry points for the GDUT experiments."""

import argparse
import gc
import hashlib
import json
from pathlib import Path
import random
import numpy as np
import torch
import ultralytics
from ultralytics import YOLO
from dataset_pipeline import find_prepared, mounted_yaml, dump
from research_trainer import ARCHITECTURE_REVISION, TRAINERS, inventory, patch
from training_recipe import optimizer_settings

PIN = "8.4.115"


def seed():
    random.seed(3407)
    np.random.seed(3407)
    torch.manual_seed(3407)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(3407)


def fingerprint(prepared):
    return hashlib.sha256((prepared / "dataset_manifest.json").read_bytes()).hexdigest()


def find_artifact(key, stage, model, explicit=""):
    files = (
        [Path(explicit) / "run_manifest.json"]
        if explicit
        else list(Path("/kaggle/input").rglob("run_manifest.json"))
    )
    found = []
    for p in files:
        m = json.loads(p.read_text())
        if (m.get("dataset"), m.get("stage"), m.get("model")) == (key, stage, model):
            found.append((p.parent, m))
    if len(found) != 1:
        raise RuntimeError(
            f"Need one {key}/{stage}/{model} input; found {[str(p) for p,m in found]}. Set explicit artifact directory."
        )
    p, m = found[0]
    weight = p / m["checkpoint"]
    if not weight.exists():
        raise FileNotFoundError(weight)
    assert (
        hashlib.sha256(weight.read_bytes()).hexdigest() == m["checkpoint_sha256"]
    ), "Checkpoint mismatch"
    return weight, m


def metrics_dict(metrics):
    return {str(k): float(v) for k, v in metrics.results_dict.items()}


def run(
    key,
    stage,
    model,
    device="0,1",
    epochs=100,
    batch=16,
    prepared_dir="",
    checkpoint_dir="",
    smoke=False,
    imgsz=640,
    recipe="legacy",
):
    assert ultralytics.__version__ == PIN, (ultralytics.__version__, PIN)
    if imgsz != 640:
        raise ValueError("The experiment uses a fixed image size of 640.")
    seed()
    prepared = find_prepared(key, prepared_dir)
    data = mounted_yaml(prepared, stage.startswith("A") and stage != "A0")
    meta = json.loads((prepared / "dataset_manifest.json").read_text())
    if (
        stage.startswith("A")
        and stage != "A0"
        and meta.get("augmentation_imgsz", 640) < imgsz
    ):
        raise ValueError(
            "Hardaug images were pre-resized below IMGSZ. Rebuild preparation with AUG_IMAGE_SIZE >= IMGSZ, then rerun Stage1 with that same prepared output."
        )
    root = Path("/kaggle/working/runs") / (key + "_" + stage + "_" + model)
    if root.exists():
        raise FileExistsError(
            f"Use a fresh session/output name; refusing to mix runs: {root}"
        )
    if stage == "S1":
        weight = model + ".pt"
        prior = None
    else:
        if model not in ("yolo11s", "yolov8s"):
            raise ValueError(
                "Preserve original Stage2 backbones: yolo11s and yolov8s only."
            )
        weight, prior = find_artifact(key, "S1", model, checkpoint_dir)
        assert prior["dataset_fingerprint"] == fingerprint(
            prepared
        ), "Stage1 used a different split/data version"
        assert not prior.get(
            "smoke"
        ), "Cannot use a smoke-test checkpoint for full experiments"
        assert (
            prior["names"] == meta["names"]
        ), "Stage1 class mapping differs from the prepared dataset"
    if stage == "A0":
        imgsz = int(prior.get("imgsz", 640))
        if imgsz != 640:
            raise ValueError("Stage1 image size differs from the experiment protocol.")
        root.mkdir(parents=True)
        val = YOLO(str(weight)).val(
            data=data,
            split="val",
            imgsz=imgsz,
            batch=batch,
            device=device.split(",")[0],
            project=str(root),
            name="val",
            plots=True,
        )
        dump(
            root / "evaluation.json",
            {
                "dataset": key,
                "stage": stage,
                "model": model,
                "split": "val",
                "metrics": metrics_dict(val),
                "dataset_fingerprint": fingerprint(prepared),
                "source_checkpoint_sha256": prior["checkpoint_sha256"],
                "imgsz": imgsz,
            },
        )
        return
    args = dict(
        model=str(weight),
        data=data,
        epochs=1 if smoke else epochs,
        imgsz=imgsz,
        batch=batch,
        device=device,
        workers=4,
        seed=3407,
        deterministic=True,
        patience=50,
        project=str(root.parent),
        name=root.name,
        exist_ok=False,
        save=True,
        plots=True,
        cache=False,
        amp=True,
        cos_lr=False,
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.45 if stage == "A1" else 0.4,
        mixup=0.1 if stage == "A1" else 0.0,
    )
    args.update(optimizer_settings(stage, recipe))
    initialization_check = None
    if stage in ("A2", "A3", "A5", "A6"):
        preview = YOLO(str(weight))
        patch(preview.model, stage)
        policy = preview.model.initialization_policy
        module_audit = preview.model.initialization_audit
        checks = Path("/kaggle/working/initialization_checks") / root.name
        initial = preview.val(
            data=data,
            split="val",
            imgsz=imgsz,
            batch=batch,
            device=device.split(",")[0],
            project=str(checks),
            name="before_training",
            plots=False,
        )
        initialization_check = dict(
            initialization_policy=policy,
            modules=module_audit,
            pretrain_validation=metrics_dict(initial),
            dataset_fingerprint=fingerprint(prepared),
            source_checkpoint_sha256=prior["checkpoint_sha256"],
            imgsz=imgsz,
        )
        dump(checks / "initialization_check.json", initialization_check)
        print("Validation before training:", initialization_check, flush=True)
        del preview, initial
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        seed()
    if stage == "S1":
        args.pop("model")
        YOLO(str(weight)).train(**args)
    else:
        TRAINERS[stage](overrides=args).train()
    best = root / "weights/best.pt"
    loaded = YOLO(str(best))
    if stage != "S1":
        assert (
            getattr(loaded.model, "research_variant", None) == stage
        ), "Custom model was lost in checkpoint"
        print("Checkpoint modules:", inventory(loaded.model))
        if "," in device:
            for rank in range(len(device.split(","))):
                audit = root / f"architecture_rank_{rank}.json"
                assert audit.exists(), f"Missing DDP worker audit: {audit}"
                worker = json.loads(audit.read_text())
                assert (
                    worker["modules"] == inventory(loaded.model)
                    and worker["variant"] == stage
                )
    val = loaded.val(
        data=data,
        split="val",
        imgsz=imgsz,
        batch=batch,
        device=device.split(",")[0],
        project=str(root),
        name="val_check",
        plots=True,
    )
    report = dict(
        dataset=key,
        stage=stage,
        model=model,
        names=meta["names"],
        seed=3407,
        dataset_fingerprint=fingerprint(prepared),
        checkpoint="weights/best.pt",
        checkpoint_sha256=hashlib.sha256(best.read_bytes()).hexdigest(),
        ultralytics=ultralytics.__version__,
        torch=torch.__version__,
        smoke=smoke,
        imgsz=imgsz,
        training_args=args,
        recipe=recipe,
        architecture_revision=getattr(loaded.model, "architecture_revision", "stock"),
        split_protocol=meta.get(
            "split_protocol",
            "official_test_deduplicated",
        ),
        initialization_policy=getattr(loaded.model, "initialization_policy", "stock"),
        initialization_audit=getattr(loaded.model, "initialization_audit", []),
        initialization_check=initialization_check,
        source_checkpoint_sha256=prior["checkpoint_sha256"] if prior else None,
        status=getattr(loaded.model, "research_status", "stock"),
        modules=inventory(loaded.model),
        validation=metrics_dict(val),
        protocol="fixed640_color_safe_ablation_v2",
    )
    dump(root / "run_manifest.json", report)
    if prior and int(prior.get("imgsz", 640)) == imgsz:
        deltas = {
            k: report["validation"][k] - v
            for k, v in prior["validation"].items()
            if k in report["validation"]
        }
        dump(
            root / "comparison_with_stage1.json",
            {
                "deltas": deltas,
                "stage1_recipe": prior.get("recipe", "legacy"),
                "current_recipe": recipe,
            },
        )
        if deltas.get("metrics/mAP50-95(B)", 0) < 0:
            print(
                "WARNING: validation mAP50-95 decreased vs Stage1. Inspect before continuing all ablations."
            )
    del loaded
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def final_test(key, prepared_dir="", device="0", batch=16):
    import csv

    prepared = find_prepared(key, prepared_dir)
    data = mounted_yaml(prepared, False)
    rows = []
    seen = set()
    manifests = sorted(Path("/kaggle/input").rglob("run_manifest.json"))
    for p in manifests:
        m = json.loads(p.read_text())
        if m["dataset"] != key:
            continue
        assert m["dataset_fingerprint"] == fingerprint(prepared)
        assert not m.get("smoke"), "Remove smoke outputs from final evaluation inputs"
        ident = (m["stage"], m["model"])
        assert ident not in seen, f"Duplicate run {ident}"
        seen.add(ident)
        w = p.parent / m["checkpoint"]
        assert hashlib.sha256(w.read_bytes()).hexdigest() == m["checkpoint_sha256"]
        eval_imgsz = int(m.get("imgsz", 640))
        if eval_imgsz != 640:
            raise ValueError("Test image size must be 640.")
        if (
            m["model"] == "yolov8s"
            and m["stage"] in ("A5", "A6")
            and m.get("architecture_revision") != ARCHITECTURE_REVISION
        ):
            raise ValueError("Use the new YOLOv8 A5/A6 checkpoints with BiFormer.")
        y = YOLO(str(w))
        result = y.val(
            data=data,
            split="test",
            imgsz=eval_imgsz,
            batch=batch,
            device=device,
            project="/kaggle/working/test_evaluation",
            name="_".join(ident),
            plots=True,
        )
        rows.append(
            dict(
                stage=m["stage"],
                model=m["model"],
                status=m["status"],
                imgsz=eval_imgsz,
                recipe=m.get("recipe", "legacy"),
                architecture_revision=m.get("architecture_revision", "stock"),
                initialization_policy=m.get("initialization_policy", "legacy_random"),
                **metrics_dict(result),
            )
        )
        per_class = [
            dict(
                class_id=int(i),
                name=m["names"][i],
                AP50=float(result.box.all_ap[j, 0]),
                AP50_95=float(result.box.all_ap[j].mean()),
            )
            for j, i in enumerate(result.box.ap_class_index)
        ]
        dump(
            Path("/kaggle/working/test_evaluation")
            / ("_".join(ident) + "_per_class.json"),
            per_class,
        )
    required = {("S1", x) for x in ["yolov8s", "yolo11s"]} | {
        (f"A{i}", x) for i in range(1, 7) for x in ["yolo11s", "yolov8s"]
    }
    missing = required - seen
    if missing:
        print("PARTIAL collection; missing required runs:", sorted(missing))
    if not rows:
        raise RuntimeError("No run manifests mounted")
    target = Path("/kaggle/working/test_evaluation/summary.csv")
    with target.open("w", newline="") as f:
        writer = csv.DictWriter(
            f, fieldnames=sorted(set().union(*(r.keys() for r in rows)))
        )
        writer.writeheader()
        writer.writerows(rows)
    dump(
        target.with_suffix(".coverage.json"),
        {"missing": sorted(missing), "complete": not missing},
    )


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--dataset", required=True, choices=["gdut"])
    p.add_argument("--stage", required=True)
    p.add_argument("--model", default="yolo11s")
    p.add_argument("--device", default="0,1")
    p.add_argument("--epochs", type=int, default=100)
    p.add_argument("--batch", type=int, default=16)
    p.add_argument("--prepared-dir", default="")
    p.add_argument("--checkpoint-dir", default="")
    p.add_argument("--smoke", action="store_true")
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--recipe", choices=["legacy", "conservative_v2"], default="legacy")
    a = p.parse_args()
    if a.stage == "TEST":
        final_test(a.dataset, a.prepared_dir, a.device.split(",")[0], a.batch)
    else:
        run(
            a.dataset,
            a.stage,
            a.model,
            a.device,
            a.epochs,
            a.batch,
            a.prepared_dir,
            a.checkpoint_dir,
            a.smoke,
            a.imgsz,
            a.recipe,
        )


In [ ]:
%%writefile /kaggle/working/research_code/sitecustomize.py
"""Apply the notebook console format in distributed training workers."""

import os

if "LOCAL_RANK" in os.environ:
    from plain_logging import configure_plain_logging

    configure_plain_logging()


In [ ]:
%%writefile /kaggle/working/research_code/training_recipe.py
"""Optimizer and learning-rate settings for Stage1 and ablation runs."""


def optimizer_settings(stage, recipe):
    if recipe == "legacy":
        return {} if stage == "S1" else dict(optimizer="MuSGD", lr0=0.01, momentum=0.9)
    if recipe == "conservative_v2":
        return dict(
            optimizer="AdamW",
            lr0=0.0003 if stage == "S1" else 0.0001,
            momentum=0.9,
            warmup_bias_lr=0.0,
            warmup_epochs=3.0,
        )
    raise ValueError(f"Unknown recipe: {recipe}")


In [ ]:
from plain_logging import configure_plain_logging

configure_plain_logging()


module_hashes = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in CODE_DIR.glob("*.py")
}
(CODE_DIR / "source_hashes.json").write_text(
    json.dumps(module_hashes, indent=2), encoding="utf-8"
)


In [ ]:
from run_experiment import final_test

DATASET = "gdut"
PREPARED_DIR = ""
DEVICE = "0"
BATCH_SIZE = 16

final_test(DATASET, prepared_dir=PREPARED_DIR, device=DEVICE, batch=BATCH_SIZE)


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

OUTPUT_DIR = Path("/kaggle/working/test_evaluation")
INPUT_ROOT = Path("/kaggle/input")
metric_labels = {
    "metrics/precision(B)": "Precision",
    "metrics/recall(B)": "Recall",
    "metrics/mAP50(B)": "mAP50",
    "metrics/mAP50-95(B)": "mAP50-95",
}
run_keys = ["stage", "model"]
test_scores = pd.read_csv(OUTPUT_DIR / "summary.csv", float_precision="round_trip")
evaluated = set(test_scores[run_keys].itertuples(index=False, name=None))
val_rows = []
for manifest_path in sorted(INPUT_ROOT.rglob("run_manifest.json")):
    run = json.loads(manifest_path.read_text(encoding="utf-8"))
    identity = (run.get("stage"), run.get("model"))
    if run.get("dataset") != DATASET or identity not in evaluated:
        continue
    scores = run.get("validation", {})
    if not all(key in scores and np.isfinite(scores[key]) for key in metric_labels):
        raise ValueError(f"Missing or invalid saved validation metrics in {manifest_path}")
    val_rows.append({
        "stage": identity[0],
        "model": identity[1],
        **{key: scores[key] for key in metric_labels},
        "source_manifest": str(manifest_path),
        "checkpoint_sha256": run["checkpoint_sha256"],
    })

val_scores = pd.DataFrame(val_rows)
if len(val_scores) != len(evaluated) or val_scores.duplicated(run_keys).any():
    raise ValueError("Expected one saved validation record for each tested checkpoint.")
val_scores.to_csv(OUTPUT_DIR / "validation_summary.csv", index=False)

def show_scores(frame, title):
    table = frame.loc[:, [*run_keys, *metric_labels]].copy()
    table["stage"] = table["stage"].replace({"S1": "A0 (Stage1)"})
    table = table.sort_values(run_keys).rename(
        columns={"stage": "Stage", "model": "Model", **metric_labels}
    )
    display(
        table.style.hide(axis="index")
        .set_caption(title)
        .format({label: "{:.2%}" for label in metric_labels.values()})
    )

show_scores(val_scores, "GDUT validation | Saved best-checkpoint scores")
show_scores(test_scores, "GDUT test | Final evaluation")
